# LLM evaluation

Author: Mathias Husted

**IMPORTANT:** Check each run with `ollama ps` to make sure Ollama is running on the CPU.

A large part of this notebook was written using Claude Opus 5.5 (Anthropic). The reason for this choice was a big overlap with the notebook `baseline_evaluation.ipynb`; most of the work was copying and rewiring existing functions to the LLMs' results instead of the baselines'. For time saving purposes, Claude was chosen to help bring the data into the same representations. Each cell was prompted manually, asking Claude to stay as close to the original as possible so that readability is guaranteed. The prompts were sent in sequence, so the previous prompt was saved in context. All outputs were checked for correctness and all interpretations are mine.

In [97]:
import pandas as pd
import math
import sys
sys.path.append("../shared")
from scoring import wilson_score_interval, decide
from llm_run_loader import load_run

In [98]:
BENCHMARK_PATH = "../../data/dataset/benchmark.csv"
RUNS_PATH = "../../results/llm"
RESULTS_PATH = f"{RUNS_PATH}/resolved"
FINAL_PATH = f"../../results/final"

benchmark = pd.read_csv(BENCHMARK_PATH, dtype=str, keep_default_na=False)

In [99]:
# Load the runs into memory like in the generation notebooks
arms = ['A', 'B']
models = ['gemma4_e2b', 'gemma4_e4b', 'phi4-mini_3.8b', 'qwen3.5_4b', 'Google Gemini'] # As per file name in /runs
runs_data = [
    {
        'arm': arm,
        'model': model,
        'path': f"{RUNS_PATH}/eco-id-results-{arm}-{model}.csv"
    }
    for arm in arms
    for model in models
]

# Think run
runs_data.append({
    'arm': 'B',
    'model': 'gemma4_e4b_think',
    'path': f"{RUNS_PATH}/eco-id-results-B-gemma4_e4b_think.csv"
})


# Check file rows match up with benchmark after run
# Then add the ID column to the run data


for run in runs_data:
    run['df'] = load_run(run['path'], run['arm'], benchmark)

run_lookup = {(run['arm'], run['model']): run['df'] for run in runs_data}


# Load the API-resolved runs
resolved_data = []
for model in models + ['gemma4_e4b_think']:
    path = f"{RESULTS_PATH}/resolved_B_{model}.csv" 
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert len(df) == len(benchmark)
    assert df["id"].is_unique
    resolved_data.append({'arm': 'B', 'model': model, 'df': df})

# Arm B

In [100]:
# Claude Opus 5.5 (Anthropic) wrote this cell (14.10.2026)
# Prompt: Based on the logic in baseline_evaluation.ipynb and the variable loaded into memory, create the next cell
#         handling the data merge of Arm B only. It should stay as close to the existing code as possible and only perform the
#         same functions as the original cell.
# Inputs: - This file up to this cell (previous cells in context)
#         - baseline_evaluation.ipynb
#         - The llm results folder containing the CSVs


gold_columns = ["id", "input", "canonical_name", "entity_type", "noise_type", "id_status", "accepted_id"]


for item in resolved_data:
    run_df = None
    for run in runs_data:
        if run['arm'] == item['arm'] and run['model'] == item['model']:
            run_df = run['df']
    assert run_df is not None

    llm = run_df[["id", "Resolved Name", "Entity Type", "Status"]].rename(
        columns={"Resolved Name": "pred_name", "Entity Type": "pred_entity_type_run", "Status": "pred_status"})
    res = item['df'][["id", "query", "pred_entity_type", "n_candidates", "returned_id", "candidate_ids", "matched_field"]]

    merged_fields = pd.merge(res, llm, on="id", validate="one_to_one")
    assert (merged_fields["query"] == merged_fields["pred_name"]).all(), f"For model {item['model']}: All names remain unchanged by LLM. Check the file"
    assert (merged_fields["pred_entity_type"] == merged_fields["pred_entity_type_run"]).all()

    item['merged'] = pd.merge(merged_fields, benchmark[gold_columns], on="id", how="left", validate="one_to_one")
    item['merged']["n_candidates"] = item['merged']["n_candidates"].astype(int)
    assert len(item['merged']) == len(benchmark)

for item in resolved_data:
    merged_fields = item['merged']

    merged_fields["outcome"] = merged_fields.apply(decide, axis=1)
    merged_fields["llm_abstain"] = merged_fields["matched_field"] == "llm_abstain"
    merged_fields["inconsistent_output"] = (merged_fields["pred_status"] == "abstain") & (merged_fields["pred_name"] != "")

    item['merged'] = merged_fields

In [101]:
# Claude Opus 5.5 (Anthropic) wrote this cell (14.10.2026)
# Prompt: Create the next cell handling the combination of the results into a single dataframe
# Inputs: - Context from previous prompt

# Combine all runs into one dataframe
all_results = pd.concat(
    [
        item["merged"].assign(
            arm=item["arm"],
            model=item["model"],
            arm_model=f"{item['arm']}_{item['model']}"
        )
        for item in resolved_data
    ],
    ignore_index=True
)

# Added by me
all_results.to_csv(f"{FINAL_PATH}/llm_arm_b.csv", index=False)

outcome_ct = pd.crosstab(all_results["outcome"], all_results["arm_model"])

outcome_ct

arm_model,B_Google Gemini,B_gemma4_e2b,B_gemma4_e4b,B_gemma4_e4b_think,B_phi4-mini_3.8b,B_qwen3.5_4b
outcome,,,,,,
abstain_multiple,0,4,1,2,3,1
abstain_no_match,6,47,28,41,39,46
abstained_on_none,15,13,14,17,13,15
correct,273,221,247,231,219,221
returned_on_none,3,5,4,1,5,3
wrong_entity,22,29,25,27,40,33


In [102]:
# Claude Opus 5.5 (Anthropic) wrote this cell (14.10.2026)
# Prompt: Create the next cell handling the creation of the headline table
# Inputs: - Context from previous prompt

# Accuracy = correct / n
# Coverage = (correct + wrong) / n
# Precision = correct / (correct + wrong)
llm_single = all_results[all_results['id_status'] == "single"]

llm_grouped = llm_single.groupby(['entity_type', 'arm_model'])

for key, group in llm_grouped:
    assert len(group) == (106 if key[0] == "chemical" else 119), key

llm_headline = {
    "Entity type": [key[0] for key, group in llm_grouped],
    "Method": [key[1] for key, group in llm_grouped],
    "n": [len(group) for key, group in llm_grouped],
    "Correct": [(group['outcome'] == "correct").sum() for key, group in llm_grouped],
    "Accuracy": [round((group['outcome'] == "correct").sum()/len(group), 3) for key, group in llm_grouped],
    "95% CI": [tuple(round(x, 3) for x in wilson_score_interval((group['outcome'] == "correct").sum(), len(group))) for key, group in llm_grouped],
    "Coverage": [round(group['outcome'].isin(["correct", "wrong_entity"]).sum() / len(group), 3) for key, group in llm_grouped],
    "Precision": [(round((group['outcome'] == "correct").sum() / group['outcome'].isin(["correct", "wrong_entity"]).sum(), 3)
                   if group['outcome'].isin(["correct", "wrong_entity"]).sum() > 0 else "n/a") for key, group in llm_grouped],
    "Wrong entity": [(group['outcome'] == "wrong_entity").sum() for key, group in llm_grouped],
}

llm_headline = pd.DataFrame(data=llm_headline)
llm_headline

,Entity type,Method,n,Correct,Accuracy,95% CI,Coverage,Precision,Wrong entity
0,chemical,B_Google Gemini,106,104,0.981,"(0.934, 0.995)",1.000,0.981,2
1,chemical,B_gemma4_e2b,106,96,0.906,"(0.835, 0.948)",0.972,0.932,7
2,chemical,B_gemma4_e4b,106,96,0.906,"(0.835, 0.948)",0.972,0.932,7
3,chemical,B_gemma4_e4b_think,106,95,0.896,"(0.824, 0.941)",0.943,0.950,5
4,chemical,B_phi4-mini_3.8b,106,94,0.887,"(0.812, 0.934)",0.972,0.913,9
5,chemical,B_qwen3.5_4b,106,97,0.915,"(0.846, 0.955)",0.981,0.933,7
6,gene,B_Google Gemini,119,115,0.966,"(0.917, 0.987)",0.983,0.983,2
7,gene,B_gemma4_e2b,119,81,0.681,"(0.592, 0.758)",0.748,0.910,8
8,gene,B_gemma4_e4b,119,106,0.891,"(0.822, 0.935)",0.899,0.991,1
9,gene,B_gemma4_e4b_think,119,92,0.773,"(0.69, 0.839)",0.798,0.968,3


In [103]:
# Claude Opus 5.5 (Anthropic) wrote this cell (14.10.2026)
# Prompt: Create the next cell handling the creation of the noise type-grouped accuracy table
# Inputs: - Context from previous prompt

# Per-class table, single rows. One column per run (arm_model).
llm_single = all_results[all_results['id_status'] == "single"]

run_names = []
for name in llm_single['arm_model']:
    if name not in run_names:
        run_names.append(name)

def calc_accuracy_ci(group):
    correct_count = (group['outcome'] == "correct").sum()
    low, high = wilson_score_interval(correct_count, len(group))
    return f"{correct_count}/{len(group)} {100 * correct_count / len(group):.1f} [{100 * low:.1f}–{100 * high:.1f}]"

class_grouped = llm_single.groupby(['entity_type', 'noise_type'])

table_rows = []
wrong_rows = []
for key, group in class_grouped:
    entity_type, noise_type = key
    n_class = len(group[group['arm_model'] == run_names[0]])
    row_acc = [entity_type, noise_type, n_class]
    row_wrong = [entity_type, noise_type, n_class]
    for run_name in run_names:
        run_group = group[group['arm_model'] == run_name]
        assert len(run_group) == n_class, (key, run_name)
        row_acc.append(calc_accuracy_ci(run_group))
        row_wrong.append((run_group['outcome'] == "wrong_entity").sum())
    table_rows.append(row_acc)
    wrong_rows.append(row_wrong)

class_table = pd.DataFrame(table_rows, columns=["Entity type", "Noise type", "n"] + run_names)
class_wrong = pd.DataFrame(wrong_rows, columns=["Entity type", "Noise type", "n"] + run_names)

class_table

,Entity type,Noise type,n,B_gemma4_e2b,B_gemma4_e4b,B_phi4-mini_3.8b,B_qwen3.5_4b,B_Google Gemini,B_gemma4_e4b_think
0,chemical,abbreviation,12,11/12 91.7 [64.6–98.5],9/12 75.0 [46.8–91.1],11/12 91.7 [64.6–98.5],11/12 91.7 [64.6–98.5],12/12 100.0 [75.7–100.0],10/12 83.3 [55.2–95.3]
1,chemical,casing,13,13/13 100.0 [77.2–100.0],13/13 100.0 [77.2–100.0],13/13 100.0 [77.2–100.0],13/13 100.0 [77.2–100.0],13/13 100.0 [77.2–100.0],13/13 100.0 [77.2–100.0]
2,chemical,separator,12,11/12 91.7 [64.6–98.5],12/12 100.0 [75.7–100.0],11/12 91.7 [64.6–98.5],12/12 100.0 [75.7–100.0],12/12 100.0 [75.7–100.0],11/12 91.7 [64.6–98.5]
3,chemical,synonym,24,22/24 91.7 [74.2–97.7],22/24 91.7 [74.2–97.7],24/24 100.0 [86.2–100.0],22/24 91.7 [74.2–97.7],24/24 100.0 [86.2–100.0],22/24 91.7 [74.2–97.7]
4,chemical,tool_artifact,10,7/10 70.0 [39.7–89.2],7/10 70.0 [39.7–89.2],6/10 60.0 [31.3–83.2],8/10 80.0 [49.0–94.3],9/10 90.0 [59.6–98.2],6/10 60.0 [31.3–83.2]
5,chemical,typo,18,15/18 83.3 [60.8–94.2],17/18 94.4 [74.2–99.0],14/18 77.8 [54.8–91.0],16/18 88.9 [67.2–96.9],17/18 94.4 [74.2–99.0],16/18 88.9 [67.2–96.9]
6,chemical,unperturbed,7,7/7 100.0 [64.6–100.0],6/7 85.7 [48.7–97.4],7/7 100.0 [64.6–100.0],6/7 85.7 [48.7–97.4],7/7 100.0 [64.6–100.0],7/7 100.0 [64.6–100.0]
7,chemical,word_order,10,10/10 100.0 [72.2–100.0],10/10 100.0 [72.2–100.0],8/10 80.0 [49.0–94.3],9/10 90.0 [59.6–98.2],10/10 100.0 [72.2–100.0],10/10 100.0 [72.2–100.0]
8,gene,abbreviation,21,16/21 76.2 [54.9–89.4],21/21 100.0 [84.5–100.0],18/21 85.7 [65.4–95.0],20/21 95.2 [77.3–99.2],21/21 100.0 [84.5–100.0],21/21 100.0 [84.5–100.0]
9,gene,casing,11,10/11 90.9 [62.3–98.4],11/11 100.0 [74.1–100.0],10/11 90.9 [62.3–98.4],10/11 90.9 [62.3–98.4],11/11 100.0 [74.1–100.0],11/11 100.0 [74.1–100.0]


# Arm A

In [104]:
# Claude Opus 5.5 (Anthropic) wrote this cell (14.10.2026)
# Prompt: Create the next cell handling the creation of the merged dataframe for arm A. Merge on id.
# Inputs: - Context from previous prompt

checked_data = []
for model in models:
    path = f"{RESULTS_PATH}/resolved_A_{model}.csv"   # use your real arm A filename
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert len(df) == len(benchmark) and df["id"].is_unique
    assert "name_in_entry" in df.columns              # schema check: arm A, not arm B
    checked_data.append({'arm': 'A', 'model': model, 'df': df})

for item in checked_data:
    run_df = run_lookup[('A', item['model'])]
    llm = run_df[["id", "Status", "Identifier", "Resolved Name"]].rename(
        columns={"Status": "pred_status", "Identifier": "identifier_run", "Resolved Name": "pred_name_run"})
    chk = item['df'][["id", "pred_name", "pred_entity_type", "identifier", "id_format", "exists", "entry_status", "name_in_entry"]]
    a_m = pd.merge(chk, llm, on="id", validate="one_to_one")
    assert (a_m["identifier"] == a_m["identifier_run"]).all(), f"{item['model']}: stale file"
    assert (a_m["pred_name"] == a_m["pred_name_run"]).all(), f"{item['model']}: stale file"
    item['merged'] = pd.merge(a_m, benchmark[gold_columns], on="id", how="left", validate="one_to_one")
    assert len(item['merged']) == len(benchmark)

In [105]:
# Claude Opus 5.5 (Anthropic) wrote this cell (14.10.2026)
# Prompt: Create the next cell handling the application of the decision matrix on to the merged dataframe.
# Inputs: - Context from previous prompt
#         - Decision matrix as shown


def arm_a_outcome(row):
    resolved = row["pred_status"] != "abstain"            # status governs abstention
    if row["id_status"] == "none":                        # none rows first
        return "returned_on_none" if resolved else "abstained_on_none"
    if not resolved:
        return "abstained"
    if row["id_format"] in ["malformed", "abstain"]:
        return "malformed"
    if row["exists"] == "False":
        return "fabricated"
    if row["name_in_entry"] == "False":
        return "inconsistent"
    if row["identifier"] in row["accepted_id"].split(";"):
        return "correct"
    return "wrong_entity"

for item in checked_data:
    a_m = item['merged']
    a_m["outcome"] = a_m.apply(arm_a_outcome, axis=1)
    a_m["gold_match"] = a_m.apply(lambda r: r["id_status"] != "none" and r["identifier"] in r["accepted_id"].split(";"), axis=1)
    a_m["inconsistent_output"] = (a_m["pred_status"] == "abstain") & ((a_m["pred_name"] != "") | (a_m["identifier"] != ""))
    item['merged'] = a_m

a_results = pd.concat(
    [item["merged"].assign(model=item["model"]) for item in checked_data], ignore_index=True)

a_results.to_csv(f"{FINAL_PATH}/llm_arm_a.csv", index=False)

In [106]:
# Claude Opus 5.5 (Anthropic) wrote this cell (14.10.2026)
# Prompt: Create the next cell handling the arm A headline table
# Inputs: - Context from previous prompt
# Added ECO-ID checks

a_single = a_results[a_results['id_status'] == "single"]
a_grouped = a_single.groupby(['entity_type', 'model'])

for key, group in a_grouped:
    assert len(group) == (106 if key[0] == "chemical" else 119), key

def cnt(group, label):
    return (group['outcome'] == label).sum()

returned = ["correct", "inconsistent", "wrong_entity", "fabricated"]

arm_a_headline = pd.DataFrame({
    "Entity type": [key[0] for key, group in a_grouped],
    "Model": [key[1] for key, group in a_grouped],
    "n": [len(group) for key, group in a_grouped],
    "Correct": [cnt(group, "correct") for key, group in a_grouped],
    "Accuracy": [round(cnt(group, "correct") / len(group), 3) for key, group in a_grouped],
    "95% CI": [tuple(round(x, 3) for x in wilson_score_interval(cnt(group, "correct"), len(group))) for key, group in a_grouped],
    "Fabricated": [cnt(group, "fabricated") for key, group in a_grouped],
    "Inconsistent": [cnt(group, "inconsistent") for key, group in a_grouped],
    "Wrong entity": [cnt(group, "wrong_entity") for key, group in a_grouped],
    "Malformed": [cnt(group, "malformed") for key, group in a_grouped],
    "Abstained": [cnt(group, "abstained") for key, group in a_grouped],
    # ECO-ID existence filter: everything that exists passes
    "Passes existence": [group['outcome'].isin(["correct", "inconsistent", "wrong_entity"]).sum() for key, group in a_grouped],
    "Passes but not correct": [group['outcome'].isin(["inconsistent", "wrong_entity"]).sum() for key, group in a_grouped],
})
arm_a_headline

,Entity type,Model,n,Correct,Accuracy,95% CI,Fabricated,Inconsistent,Wrong entity,Malformed,Abstained,Passes existence,Passes but not correct
0,chemical,Google Gemini,106,92,0.868,"(0.79, 0.92)",3,11,0,0,0,103,11
1,chemical,gemma4_e2b,106,0,0.000,"(0.0, 0.035)",27,71,0,8,0,71,71
2,chemical,gemma4_e4b,106,2,0.019,"(0.005, 0.066)",14,87,0,0,3,89,87
3,chemical,phi4-mini_3.8b,106,0,0.000,"(0.0, 0.035)",2,72,0,31,1,72,72
4,chemical,qwen3.5_4b,106,2,0.019,"(0.005, 0.066)",13,91,0,0,0,93,91
5,gene,Google Gemini,119,104,0.874,"(0.802, 0.922)",4,11,0,0,0,115,11
6,gene,gemma4_e2b,119,0,0.000,"(0.0, 0.031)",20,94,0,1,4,94,94
7,gene,gemma4_e4b,119,4,0.034,"(0.013, 0.083)",23,69,0,0,23,73,69
8,gene,phi4-mini_3.8b,119,0,0.000,"(0.0, 0.031)",25,87,0,7,0,87,87
9,gene,qwen3.5_4b,119,0,0.000,"(0.0, 0.031)",24,71,0,18,6,71,71


In [107]:
# Claude Opus 5.5 (Anthropic) wrote this cell (14.10.2026)
# Prompt: Create the next cell handling abstain counts
# Inputs: - Context from previous prompt
pd.crosstab(a_results[a_results['id_status'] == "none"]['outcome'], a_results[a_results['id_status'] == "none"]['model'])

# Abstain counts vs inventory (e2b 5, e4b 41, phi 3, Gemini 4); use the status column, not outcome
#print(a_results.groupby('model')['pred_status'].apply(lambda s: (s == "abstain").sum()))
#print(a_results.groupby('model')['inconsistent_output'].sum())   # the contradictory rows you listed

model,Google Gemini,gemma4_e2b,gemma4_e4b,phi4-mini_3.8b,qwen3.5_4b
outcome,,,,,
abstained_on_none,3,1,9,2,8
returned_on_none,15,17,9,16,10


### Final check: Malformed rows.

This part was not written by Claude.

After having checked the malformed rows with status `single` for each model, we can observe some tendencies of each model that causes the ID format to come back as malformed despite having output an ID. For example, Qwen puts consistently puts the gene ID in square brackets on the malformed rows. For each model, we can try and correct for those tendencies and see if it has an effect on accuracy.

We will perform a simple check: For each error type, we will check for a clear pattern, in which case we will extract the ID and see if it appears in the accepted ID list of that chemical.

In [108]:
# Let's check malformed single ID rows

malformed = a_results[(a_results["id_status"] == "single") & (a_results["id_format"] == "malformed")]
# Phi had both chemicals and genes, so we process them separately
malformed_phi_chem = malformed[(malformed["model"] == "phi4-mini_3.8b") & (malformed["entity_type"] == "chemical")]
malformed_phi_gene = malformed[(malformed["model"] == "phi4-mini_3.8b") & (malformed["entity_type"] == "gene")]
# Qwen only had malformed gene rows
malformed_qwen = malformed[malformed["model"] == "qwen3.5_4b"]
# Gemma e2b also had both chemicals and genes, so we process them separately
malformed_gemma_chem = malformed[(malformed["model"] == "gemma4_e2b") & (malformed["entity_type"] == "chemical")]
malformed_gemma_gene = malformed[(malformed["model"] == "gemma4_e2b") & (malformed["entity_type"] == "gene")]

**Total malformed rows over all runs:** 65
Of those, we can see from the headline table that Phi 4-mini on chemicals attributed the most (31) followed by Qwen 3.5 4B on genes (18). Gemma e2b also showed some errors on chemicals (8), followed by Phi 4-mini on genes (7) and finally a single wrong row of Gemma e2b on genes.

### Qwen 3.5 4B
In this case, Qwen made the same error on all malformed rows and only on **genes**: Instead of the format `HGNC:xxxx`, it output the number in square brackets: `HGNC:[xxxx]`. By simply extracting the substring, we can reconstruct the intended format and check if the ID now matches.

### Phi-4-mini
The error was uniform on all malformed **chemical** rows: Instead of simply outputting the CID, it output the format `PubChem CID: xxxxx`, sometimes with entirely made up IDs. Again, we will extract the number and check if it matches. On the malformed **gene** rows, it misidentified all of them as chemicals, so a fix won't help us here.

### Gemma 4 e2b
On **chemicals**, it made multiple types of errors: It output the ID as a float, and on the rest it output IDs with a dash e.g. `6422-91-1`, probably confusing the ID type with another database format. On **genes**, the same mistake using a dash was found again. We can't try to salvage this error type.

In [109]:
import re

def extract_id_number(rows):
    n_accepted = 0
    for idx, row in rows.iterrows():
        identifier = row["identifier"]
        number = re.search(r'[0-9]+', identifier)
        fixed = ("HGNC:" + number[0]) if row["pred_entity_type"] == "gene" else number[0]
        is_accepted = fixed in row["accepted_id"].split(";")
        if is_accepted:
            n_accepted += 1
    return n_accepted

# CHECK PHI-4-MINI ROWS
n_accepted_phi_chem = extract_id_number(malformed_phi_chem)
print(f"Phi chemical rows that are accepted after the fix: {n_accepted_phi_chem}")
n_accepted_phi_gene = extract_id_number(malformed_phi_gene)
print(f"Phi chemical rows that are accepted after the fix: {n_accepted_phi_gene}")

# CHECK QWEN 3.5 4B ROWS
n_accepted_qwen = extract_id_number(malformed_qwen)
print(f"Qwen rows that are accepted after the fix: {n_accepted_qwen}")

Phi chemical rows that are accepted after the fix: 0
Phi chemical rows that are accepted after the fix: 0
Qwen rows that are accepted after the fix: 0


### Final verdict of malformed rows

The correction of malformed rows had no effect and would not have made any positive difference in accuracy.